# Homework 3 — Machine Learning for Classification

Task: [homework.md](homework.md) · [2026 assignment](https://github.com/DataTalksClub/machine-learning-zoomcamp/blob/main/cohorts/2026/homework/03-classification/homework.md)

Predict whether a lead signs up for the platform (`converted`) using the pinned 2026 lead-scoring dataset. Run all cells in order to reproduce the answers.

In [1]:
from pathlib import Path
import hashlib
import ssl
from urllib.request import urlopen

import certifi
import numpy as np
import pandas as pd
import sklearn
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, mutual_info_score
from sklearn.model_selection import train_test_split

print(f"NumPy {np.__version__}; pandas {pd.__version__}; scikit-learn {sklearn.__version__}")

NumPy 2.5.3; pandas 3.0.6; scikit-learn 1.9.1


## Load the data

Cache the course dataset locally; the CSV is excluded from Git.

In [2]:
DATA_URL = (
    "https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/"
    "main/cohorts/2026/data/course_lead_scoring_2026.csv"
)
# Support running from either the repository root or this notebook's folder.
DATA_DIR = Path("03-classification") if Path("03-classification").is_dir() else Path(".")
data_path = DATA_DIR / "course_lead_scoring_2026.csv"
if not data_path.exists():
    context = ssl.create_default_context(cafile=certifi.where())
    with urlopen(DATA_URL, context=context, timeout=60) as response:
        data_path.write_bytes(response.read())

target = "converted"
df = pd.read_csv(data_path)
print(f"Rows: {len(df):,}")
print(f"Dataset SHA-256: {hashlib.sha256(data_path.read_bytes()).hexdigest()}")
df.head()

Rows: 5,000
Dataset SHA-256: 2d9da196bdefd2a45aa0a17ca21c19f06a5d9d56af628be1f04bbb3531cc12e9


,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


## Data preparation

Check for missing values, then fill categorical features with `'NA'` and numerical features with `0.0`.

In [3]:
categorical = [c for c in df.columns if c != target and not pd.api.types.is_numeric_dtype(df[c])]
numerical = [c for c in df.columns if c != target and pd.api.types.is_numeric_dtype(df[c])]
print("Categorical:", categorical)
print("Numerical:", numerical)
df.isna().sum()

Categorical: ['lead_source', 'industry', 'employment_status', 'location']
Numerical: ['annual_income', 'number_of_courses_viewed', 'interaction_count', 'lead_score']


lead_source                 148
industry                    240
employment_status           194
location                    208
annual_income               369
number_of_courses_viewed      0
interaction_count             0
lead_score                   35
converted                     0
dtype: int64

In [4]:
df[categorical] = df[categorical].fillna("NA")
df[numerical] = df[numerical].fillna(0.0)
assert df.isna().sum().sum() == 0
df[target].value_counts(normalize=True)

converted
1    0.5766
0    0.4234
Name: proportion, dtype: float64

## Q1. Mode for `industry`

In [5]:
df["industry"].value_counts().head()

industry
technology    1173
retail         925
healthcare     840
finance        720
education      639
Name: count, dtype: int64

**Answer: `technology`** — 1,173 rows.

## Q2. Biggest correlation

Correlation matrix of the numerical features (after imputation), then compare only the four candidate pairs.

In [6]:
corr = df[numerical].corr()
corr.round(3)

,annual_income,number_of_courses_viewed,interaction_count,lead_score
annual_income,1.000,0.161,0.123,0.229
number_of_courses_viewed,0.161,1.000,0.722,0.757
interaction_count,0.123,0.722,1.000,0.916
lead_score,0.229,0.757,0.916,1.000


In [7]:
pairs = [
    ("interaction_count", "lead_score"),
    ("number_of_courses_viewed", "lead_score"),
    ("number_of_courses_viewed", "interaction_count"),
    ("annual_income", "interaction_count"),
]
pd.DataFrame(
    [{"pair": f"{a} / {b}", "correlation": corr.loc[a, b]} for a, b in pairs]
).sort_values("correlation", key=abs, ascending=False)

,pair,correlation
0,interaction_count / lead_score,0.915746
1,number_of_courses_viewed / lead_score,0.757204
2,number_of_courses_viewed / interaction_count,0.721609
3,annual_income / interaction_count,0.122842


**Answer: `interaction_count` and `lead_score`** — correlation ≈ 0.916.

## Split the data

Use the exact calls from the assignment (60%/20%/20%), then remove `converted` from the feature frames.

In [8]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=42)

y_train = df_train.pop(target).to_numpy()
y_val = df_val.pop(target).to_numpy()
y_test = df_test.pop(target).to_numpy()
print({"train": len(df_train), "validation": len(df_val), "test": len(df_test)})

{'train': 3000, 'validation': 1000, 'test': 1000}


## Q3. Mutual information

Training set only; scores rounded to 2 decimals.

In [9]:
mi = pd.Series({c: mutual_info_score(y_train, df_train[c]) for c in categorical}, name="mi")
pd.DataFrame({"mi": mi, "mi_2dp": mi.round(2)}).sort_values("mi", ascending=False)

,mi,mi_2dp
lead_source,0.034346,0.03
employment_status,0.018966,0.02
industry,0.001995,0.00
location,0.001043,0.00


**Answer: `lead_source`** — MI 0.03.

## Q4. Logistic regression accuracy

One-hot encode categorical features with `DictVectorizer` (numerical features pass through) and fit with the prescribed parameters.

In [10]:
features = categorical + numerical


def train_and_score(columns, C=1.0):
    dv = DictVectorizer(sparse=False)
    X_train = dv.fit_transform(df_train[columns].to_dict(orient="records"))
    model = LogisticRegression(solver="liblinear", C=C, max_iter=1000, random_state=42)
    model.fit(X_train, y_train)
    X_val = dv.transform(df_val[columns].to_dict(orient="records"))
    return accuracy_score(y_val, model.predict(X_val))


base_accuracy = train_and_score(features)
print(f"Validation accuracy: {base_accuracy:.4f} -> {round(base_accuracy, 2)}")

Validation accuracy: 0.6450 -> 0.65


**Answer: 0.65** (0.645 unrounded).

## Q5. Feature elimination

Drop one feature at a time and compare against the unrounded baseline accuracy. Difference = baseline − accuracy without the feature.

In [11]:
elimination = pd.DataFrame([
    {"feature": f, "accuracy": (acc := train_and_score([c for c in features if c != f])),
     "difference": base_accuracy - acc}
    for f in features
]).sort_values("difference")
elimination

,feature,accuracy,difference
4,annual_income,0.724,-0.079
1,industry,0.645,0.000
2,employment_status,0.645,0.000
3,location,0.645,0.000
7,lead_score,0.644,0.001
5,number_of_courses_viewed,0.643,0.002
0,lead_source,0.642,0.003
6,interaction_count,0.601,0.044


In [12]:
candidates = ["lead_source", "number_of_courses_viewed", "interaction_count"]
elimination.set_index("feature").loc[candidates].sort_values("difference")

,accuracy,difference
feature,,
number_of_courses_viewed,0.643,0.002
lead_source,0.642,0.003
interaction_count,0.601,0.044


**Answer: `number_of_courses_viewed`** — difference 0.002 (vs. 0.003 for `lead_source` and 0.044 for `interaction_count`).

## Q6. Regularization strength

Train on all features with each `C`, round validation accuracy to 3 decimals, and break ties by choosing the smallest `C`.

In [13]:
regularization = pd.DataFrame([
    {"C": C, "accuracy": train_and_score(features, C=C)}
    for C in [0.000001, 0.00001, 0.0001, 0.001]
])
regularization["accuracy_3dp"] = regularization["accuracy"].round(3)
best_C = regularization.sort_values(["accuracy_3dp", "C"], ascending=[False, True]).iloc[0]["C"]
print(f"Best C: {best_C}")
regularization

Best C: 0.001


,C,accuracy,accuracy_3dp
0,0.000001,0.598,0.598
1,0.000010,0.598,0.598
2,0.000100,0.613,0.613
3,0.001000,0.645,0.645


**Answer: 0.001** — validation accuracy 0.645.